In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Fase 1 - Modelo Predictivo (Titanic)

## Objetivo
El objetivo de este notebook es entrenar un modelo de Machine Learning capaz de predecir si un pasajero sobrevivió o no al hundimiento del Titanic.

Se utilizará el dataset de Kaggle "Titanic - Machine Learning from Disaster".

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

import joblib

## Carga de datos

Se cargan los archivos `train.csv` y `test.csv` desde la carpeta `data/`.

- `train.csv`: contiene los datos de entrenamiento y la variable objetivo `Survived`.
- `test.csv`: contiene datos para generar predicciones finales.


In [18]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [19]:
print("Tamaño train:", train_df.shape)
print("Tamaño test:", test_df.shape)

Tamaño train: (891, 12)
Tamaño test: (418, 11)


## Exploración inicial

Se revisa el tipo de variables, valores faltantes y estadísticas básicas para comprender el dataset.

In [20]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [21]:
train_df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [22]:
train_df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## Definición de variable objetivo

En este problema, la variable objetivo es `Survived`, donde:

- 0 = No sobrevivió
- 1 = Sobrevivió

In [23]:
X = train_df.drop("Survived", axis=1)
y = train_df["Survived"]

print("Variables predictoras (X):", X.shape)
print("Variable objetivo (y):", y.shape)

Variables predictoras (X): (891, 11)
Variable objetivo (y): (891,)


## División del dataset

Se divide el dataset en entrenamiento y validación para evaluar el desempeño del modelo.

In [24]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

X_train: (712, 11)
X_val: (179, 11)


## Selección de variables

Se seleccionan algunas variables relevantes para el modelo.

- Variables numéricas: Pclass, Age, SibSp, Parch, Fare
- Variables categóricas: Sex, Embarked

Se excluyen columnas como Name, Ticket y Cabin para simplificar el modelo.

In [25]:
categorical_features = ["Sex", "Embarked"]
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]

## Preprocesamiento

Se aplican transformaciones para preparar los datos:

- Imputación de valores faltantes:
  - Numéricos: mediana
  - Categóricos: valor más frecuente
- Codificación One-Hot para variables categóricas.

In [26]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## Entrenamiento del modelo

Se utiliza un modelo Random Forest debido a su buen rendimiento general y facilidad de uso para clasificación.

In [28]:
model = RandomForestClassifier(n_estimators=200, random_state=42)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [29]:
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

## Evaluación

Se realizan predicciones sobre el conjunto de validación y se calcula el accuracy del modelo.

In [30]:
y_pred = clf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

Accuracy: 0.8044692737430168
              precision    recall  f1-score   support

           0       0.82      0.85      0.84       105
           1       0.77      0.74      0.76        74

    accuracy                           0.80       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.80      0.80      0.80       179



## Entrenamiento final con todos los datos

Una vez validado el modelo, se entrena nuevamente utilizando todos los datos disponibles para generar predicciones finales.

In [31]:
clf.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

## Predicción final

Se generan predicciones sobre el dataset `test.csv` para construir el archivo de entrega (`submission.csv`).

In [32]:
test_predictions = clf.predict(test_df)

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_predictions
})

submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,1
4,896,0


In [33]:
submission.to_csv("submission.csv", index=False)
print("Archivo submission.csv guardado correctamente.")

Archivo submission.csv guardado correctamente.


In [34]:
joblib.dump(clf, "modelo_titanic.pkl")
print("Modelo guardado como modelo_titanic.pkl")

Modelo guardado como modelo_titanic.pkl
